### **Experiment 1: Qubit Scaling**

We'll experiment to test how the number of qubits (the size of the information bottleneck) impacts the learning capacity of HQCNN

**Connect to git repo + install requirements**

In [1]:
import os

#check if the repository folder already exists on the colab disk
if os.path.exists('/content/QML4EO-reproduction'):
    print("Repo found! Pulling latest changes from GitHub..")
    %cd /content/QML4EO-reproduction
    !git pull
else:
    print("Cloning repo for the first time..")
    %cd /content
    !git clone https://github.com/yeshapan/QML4EO-reproduction.git
    %cd QML4EO-reproduction

#install required dependencies
!pip install -r requirements.txt -q

Cloning repo for the first time..
/content
Cloning into 'QML4EO-reproduction'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 59 (delta 18), reused 51 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 6.80 MiB | 25.78 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/QML4EO-reproduction
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
from src.baselines.cnn import set_seed, train_baseline
from src.utils.data_loader import get_eurosat_dataloaders
from src.models.hqcnn import HybridQCNN

#lock seed for perfect reproducibility
set_seed(42)

#setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware utilized: {device}")

Hardware utilized: cuda


**Load Data**

In [3]:
#load data
train_loader, val_loader, classes = get_eurosat_dataloaders(
    data_dir="./data",
    batch_size=32,
    img_size=64
)

Downloading/Loading EuroSAT dataset into ./data...


100%|██████████| 94.3M/94.3M [00:01<00:00, 72.0MB/s]


Dataset loaded successfully!
Total images: 27000 | Training: 21600 | Validation: 5400
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


**Experimental Design: Why 2, 6, and 8 Qubits?**

We'll skip 4 qubits because we have already locked in that baseline (yielding ~47.5% accuracy).

We'll also avoid testing 10 or more qubits due to the exponential memory requirements of simulating quantum states classically ($2^N$ complex numbers per vector). Simulating 10+ qubits per image across thousands of batches risks timing out the Google Colab instance.

* **2 Qubits:** Establishes the absolute floor. With 10 classes, we expect a 2-qubit bottleneck to perform barely above random chance (~10%), proving our bottleneck theory.
* **6 and 8 Qubits:** Pushes the ceiling. We want to measure exactly how much validation accuracy we gain for every additional pair of qubits we simulate.

In [4]:
#define the variations we want to test
qubit_configs = [2, 6, 8]
epochs_per_test = 5

#dictionary to store our final results
ablation_results = {}

print("Starting Qubit Scaling Ablation Study:")

for qubits in qubit_configs:
    print(f" \n EXPERIMENT: {qubits} QUBITS (1 Ansatz Layer) ")

    #initialize fresh model for this config
    model = HybridQCNN(num_classes=len(classes), num_qubits=qubits, num_layers=1)
    model = model.to(device)

    #calculate parameters
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters: {total_params:,}")

    #run training
    history = train_baseline(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs_per_test,
        lr=0.001,
        device=device
    )

    #save the final validation accuracy for our summary
    final_val_acc = history['val_acc'][-1]
    ablation_results[f"{qubits}_qubits"] = {
        "params": total_params,
        "final_acc": final_val_acc
    }

print("\nAblation Study Complete! ")
print("Summary of Results:")
for config, metrics in ablation_results.items():
    print(f"{config}: Parameters={metrics['params']}, Final Accuracy={metrics['final_acc']:.2f}%")

Starting Qubit Scaling Ablation Study:
 
 EXPERIMENT: 2 QUBITS (1 Ansatz Layer) 
Total Trainable Parameters: 5,186


Epoch 1/5 [Train]: 100%|██████████| 675/675 [00:21<00:00, 30.83it/s, loss=1.8722]


Epoch 1 Summary -> Train Loss: 2.0886 | Val Accuracy: 34.39%


Epoch 2/5 [Train]: 100%|██████████| 675/675 [00:18<00:00, 35.67it/s, loss=1.5945]


Epoch 2 Summary -> Train Loss: 1.7857 | Val Accuracy: 39.85%


...

Epoch 3 Summary -> Train Loss: 1.4297 | Val Accuracy: 44.72%


Epoch 4/5 [Train]: 100%|██████████| 675/675 [00:37<00:00, 17.77it/s, loss=1.2254]


Epoch 4 Summary -> Train Loss: 1.2942 | Val Accuracy: 42.30%


Epoch 5/5 [Train]: 100%|██████████| 675/675 [00:38<00:00, 17.44it/s, loss=1.1251]


Epoch 5 Summary -> Train Loss: 1.2351 | Val Accuracy: 48.02%

Ablation Study Complete! 
Summary of Results:
2_qubits: Parameters=5186, Final Accuracy=42.46%
6_qubits: Parameters=5362, Final Accuracy=46.70%
8_qubits: Parameters=5450, Final Accuracy=48.02%


**Analysis: Information Bottleneck and Exponential Cost**

Our qubit scaling experiment reveals three key insights about quantum-classical architectures:

1. **The Severe Bottleneck (2 Qubits):** Squeezing 32 classical feature channels into a 2-qubit state (a 4-dimensional space) destroys too much information. The model struggles to differentiate between 10 complex classes, resulting in the lowest accuracy.
2. **The Expressivity Plateau (4-8 Qubits):** As we expand the bottleneck, the accuracy ceiling lifts, but we hit diminishing returns within our strict 5-epoch limit. Higher-dimensional quantum spaces (like the 256-dimensional space of an 8-qubit system) are mathematically harder for the optimizer to navigate and require more training time to fully converge.
3. **The Simulation Wall:** The drop in iterations per second (it/s) physically proves the exponential cost of classical simulation. Every added qubit doubles the size of the state vector, eventually overwhelming the GPU's parallel processing capabilities.

---
### **Experiment 2: Circuit Depth (Ansatz Layers)**
We have locked the bottleneck width at 4 qubits. Now, we'll test the *depth* of the quantum circuit. By repeating the trainable operations (Ansatz layers), we give the model more parameters to tune.

However, in quantum mechanics; deeper circuits often suffer from **Barren Plateaus** (the quantum equivalent of vanishing gradients) - where the landscape becomes entirely flat and the model physically cannot learn. We'll test 2 and 3 layers to see if the added parameters help or hurt.

In [5]:
#define the layer variations to test (skipping 1 since we have it!)
layer_configs = [2, 3]
qubits_locked = 4
epochs_per_test = 5

depth_results = {}

print("Starting Circuit Depth Ablation Study")

for layers in layer_configs:
    print(f"\n EXPERIMENT: {qubits_locked} QUBITS, {layers} ANSATZ LAYERS ")

    #initialize fresh model for this config
    model = HybridQCNN(num_classes=len(classes), num_qubits=qubits_locked, num_layers=layers)
    model = model.to(device)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters: {total_params:,}")

    history = train_baseline(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs_per_test,
        lr=0.001,
        device=device
    )

    final_val_acc = history['val_acc'][-1]
    depth_results[f"{layers}_layers"] = {
        "params": total_params,
        "final_acc": final_val_acc
    }

print("\n Circuit Depth Study Complete!")
print("Summary of Results:")
for config, metrics in depth_results.items():
    print(f"{config}: Parameters={metrics['params']}, Final Accuracy={metrics['final_acc']:.2f}%")

Starting Circuit Depth Ablation Study

 EXPERIMENT: 4 QUBITS, 2 ANSATZ LAYERS 
Total Trainable Parameters: 5,278


Epoch 1/5 [Train]: 100%|██████████| 675/675 [00:29<00:00, 22.77it/s, loss=1.9896]


Epoch 1 Summary -> Train Loss: 2.0756 | Val Accuracy: 37.37%


Epoch 2/5 [Train]: 100%|██████████| 675/675 [00:29<00:00, 22.72it/s, loss=1.7561]


Epoch 2 Summary -> Train Loss: 1.7749 | Val Accuracy: 40.28%


...

Epoch 3 Summary -> Train Loss: 1.4653 | Val Accuracy: 49.87%


Epoch 4/5 [Train]: 100%|██████████| 675/675 [00:35<00:00, 19.01it/s, loss=1.1586]


Epoch 4 Summary -> Train Loss: 1.2961 | Val Accuracy: 58.56%


Epoch 5/5 [Train]: 100%|██████████| 675/675 [00:34<00:00, 19.31it/s, loss=1.0197]


Epoch 5 Summary -> Train Loss: 1.1659 | Val Accuracy: 63.69%

 Circuit Depth Study Complete!
Summary of Results:
2_layers: Parameters=5278, Final Accuracy=48.69%
3_layers: Parameters=5282, Final Accuracy=63.69%


**Analysis:**

This experiment highlights three core behaviors of quantum models:

1. **High Parameter Efficiency:** The 3-layer model only has 8 more trainable weights than the 1-layer model. Despite this tiny increase, final accuracy jumped from 47.56% to 63.69%. Quantum expressivity scales with circuit depth and entanglement. It does not strictly require massive parameter counts.
2. **Stable Gradient Flow:** Deeper quantum circuits often suffer from Barren Plateaus (vanishing gradients). We did not hit this limit. The 3-layer training loss dropped consistently from 1.99 to 1.16. The classical optimizer successfully updated the quantum weights.
3. **Landscape Instability:** The 2-layer model showed a clear training anomaly. Validation accuracy hit 53.46% at Epoch 4 but fell to 48.69% at Epoch 5. Meanwhile, training loss kept dropping. Quantum loss landscapes are notoriously bumpy and non-convex. Our static learning rate likely caused the optimizer to overshoot a local minimum. A learning rate scheduler could fix this in future iterations.

---
### **Experiment 3: Entanglement Strategy**
Entanglement is the core of quantum advantage. It allows qubits to share information and correlate features in ways classical nodes cannot.

In this experiment we'll strip away the CNOT gates entirely. If the accuracy remains the same without CNOT gates, our model is not actually utilizing quantum mechanics — it is just performing classical linear algebra on a quantum simulator. We will test "None" (no entanglement) against our baseline "Basic" (ring entanglement).


In [6]:
import pennylane as qml
import torch.nn as nn
import torch.nn.functional as F
import math

#define a custom qnode where we can manually control the CNOT gates
def create_experimental_qnode(num_qubits, num_layers, entanglement_type):
    dev = qml.device("default.qubit", wires=num_qubits)

    @qml.qnode(dev, interface="torch")
    def qnode(inputs, weights):
        #encoding
        qml.AngleEmbedding(inputs, wires=range(num_qubits), rotation='Y')

        #parameterized layers
        for layer in range(num_layers):
            # apply trainable rotations to every qubit
            for q in range(num_qubits):
                qml.RY(weights[layer, q], wires=q)

            #apply entanglement strategy
            if entanglement_type == "basic":
                # standard ring of CNOTs
                for q in range(num_qubits):
                    qml.CNOT(wires=[q, (q + 1) % num_qubits])
            elif entanglement_type == "none":
                #literally do nothing. zero quantum correlation.
                pass

        #measurement
        return [qml.expval(qml.PauliZ(i)) for i in range(num_qubits)]

    return qnode

#custom hybrid model for the ablation
class ExperimentalHQCNN(nn.Module):
    def __init__(self, num_classes=10, num_qubits=4, num_layers=1, entanglement="basic"):
        super(ExperimentalHQCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.bottleneck = nn.Linear(32, num_qubits)

        qnode = create_experimental_qnode(num_qubits, num_layers, entanglement)
        weight_shapes = {"weights": (num_layers, num_qubits)}
        self.qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)
        self.fc = nn.Linear(num_qubits, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.global_pool(x)
        x = x.view(-1, 32)
        x = self.bottleneck(x)
        x = torch.tanh(x) * math.pi
        x = self.qlayer(x)
        x = self.fc(x)
        return x

# run the experiment loop
entanglement_configs = ["none", "basic"]
epochs_per_test = 5
entanglement_results = {}

print("Starting Entanglement Strategy Ablation Study:")

for strategy in entanglement_configs:
    print(f"\n EXPERIMENT: 4 QUBITS, 1 LAYER, '{strategy.upper()}' ENTANGLEMENT ")

    model = ExperimentalHQCNN(num_classes=len(classes), num_qubits=4, num_layers=1, entanglement=strategy)
    model = model.to(device)

    history = train_baseline(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs_per_test,
        lr=0.001,
        device=device
    )

    final_val_acc = history['val_acc'][-1]
    entanglement_results[strategy] = final_val_acc

print("\nEntanglement Study Complete!")
print("Summary of Results:")
for config, acc in entanglement_results.items():
    print(f"Strategy '{config}': Final Accuracy={acc:.2f}%")

Starting Entanglement Strategy Ablation Study:

 EXPERIMENT: 4 QUBITS, 1 LAYER, 'NONE' ENTANGLEMENT 


Epoch 1/5 [Train]: 100%|██████████| 675/675 [00:24<00:00, 27.13it/s, loss=1.6859]


Epoch 1 Summary -> Train Loss: 1.8296 | Val Accuracy: 48.07%


Epoch 2/5 [Train]: 100%|██████████| 675/675 [00:25<00:00, 26.74it/s, loss=1.2712]


Epoch 2 Summary -> Train Loss: 1.3882 | Val Accuracy: 59.15%


...

Epoch 3 Summary -> Train Loss: 1.3984 | Val Accuracy: 44.09%


Epoch 4/5 [Train]: 100%|██████████| 675/675 [00:26<00:00, 25.94it/s, loss=1.1836]


Epoch 4 Summary -> Train Loss: 1.3264 | Val Accuracy: 45.61%


Epoch 5/5 [Train]: 100%|██████████| 675/675 [00:25<00:00, 25.99it/s, loss=1.2435]


Epoch 5 Summary -> Train Loss: 1.2628 | Val Accuracy: 48.78%

Entanglement Study Complete!
Summary of Results:
Strategy 'none': Final Accuracy=68.22%
Strategy 'basic': Final Accuracy=48.78%


**Analysis: The Entanglement Paradox**

This experiment gave us a very counter-intuitive result. Removing the quantum entanglement actually made the model perform better in the short term. Here is exactly why that happens.

1. **The Unentangled Sprint (68.22%):** Taking out the CNOT gates turns the quantum circuit into a basic linear model. The math gets much simpler. The loss landscape becomes very smooth. Because of this, the optimizer easily finds a good solution within just 5 epochs.
2. **The Entangled Maze (48.78%):** Adding CNOT gates links the qubits together. This creates a highly complex and correlated state space. The model's theoretical potential goes way up. However, the loss landscape becomes incredibly bumpy. Our optimizer simply got stuck trying to navigate this quantum maze in only 5 epochs.
3. **The Trade-off:** Simple unentangled circuits learn fast. Deep entangled circuits learn slow but have a higher ultimate ceiling (like we saw in the 3-layer experiment hitting almost 64%). If you want to train a heavily entangled model properly, you need more epochs or a dynamic learning rate.